In [14]:
import pandas as pd
from pathlib import Path

import sys
sys.path.append("../../")

from tcremp.utils import load_analysis_repertoire
from mir.common.segments import SegmentLibrary

In [4]:
data_dir = Path("/projects/immunestatus/emerson_hla/airr_format")

metadata = pd.read_csv(data_dir / "metadata.tsv", sep='\t')

In [5]:
tcremp_dir = Path("/projects/immunestatus/emerson_hla/tcremp")

In [15]:
lib = SegmentLibrary.load_default(genes='TRB', organisms='HomoSapiens')
rep = load_analysis_repertoire('/projects/immunestatus/emerson_hla/airr_format/HIP00640.tsv', 
                               lib, 'TRB', None, 5, 30)


In [16]:
rep

Repertoire of 0 clonotypes and 0 cells:

{'path': '/projects/immunestatus/emerson_hla/airr_format/HIP00640.tsv'}
...

In [6]:
import numpy as np
np.random.seed(42)


In [11]:
def process(sample):
    print(f'started {sample}')
    emb_df = pd.read_parquet(tcremp_dir / f'{sample}_embeddings.parquet')
    desc_df = pd.read_csv(data_dir / f'{sample}.tsv', sep='\t')
    desc_df = desc_df[~desc_df.v_call.isin(['TRBV7-5*01', 'TRBV22-1*01'])].reset_index(drop=True)
    print(sample, len(emb_df), len(desc_df))
    assert len(desc_df) == len(emb_df)

    n = len(emb_df) // 3  
    random_idx = np.random.choice(emb_df.index, size=n, replace=False)

    emb_df = emb_df.loc[random_idx].reset_index(drop=True)
    desc_df = desc_df.loc[random_idx].reset_index(drop=True)
    desc_df['count'] = 1
    print(f'finished {sample}')
    return desc_df, emb_df

In [12]:
import pandas as pd
import numpy as np
import multiprocessing

for hla in metadata.allele.unique():
    print(hla)
    samples_desc = []
    samples_emb = []

    with multiprocessing.Pool(15) as p:
        res = p.map(process, metadata[metadata.allele != hla].sample_id)

    for i1, i2 in res:
        samples_desc.append(i1)
        samples_emb.append(i2)

        # Объединяем и сохраняем
    print(len(pd.concat(samples_desc)))
    pd.concat(samples_desc).reset_index(drop=True).to_csv(data_dir / f'joint_non_{hla}.tsv', sep='\t', index=False)
    pd.concat(samples_emb).reset_index(drop=True).to_parquet(tcremp_dir / f'joint_non_{hla}_embeddings.parquet')


HLA-A*01
started HIP00825started HIP14018started HIP12538started HIP14074started HIP00640started HIP10564started HIP14209started HIP08230started HIP04576started HIP13722started HIP14911started HIP09159started HIP12143started HIP00826
started HIP13871













HIP09159 102829 102842
started HIP14230
HIP13722 127208 127227
started HIP10376
HIP14209 129235 129254
started HIP14077
HIP00826 166402 166426
started HIP05409
HIP13871 155344 155365
started HIP05578
HIP00640 197164 197210
HIP08230 189035 189059
HIP12143 202423 202439
HIP14911 192628 192655
HIP04576 215556 215600
HIP10564 226787 226810
HIP14018 236833 236866
HIP12538 248035 248084
HIP14230 167175 167192
HIP10376 108807 108828
HIP14074 298398 298449
HIP00825 310820 310873
HIP14077 221883 221914
HIP05578 243203 243239
HIP05409 296570 296598


AssertionError: 

In [10]:
import sys
sys.path.append('/home/evlasova/mirpy')
from mir.common.clonotype_dataset import ClonotypeDataset
from mir.common.clonotype import ClonotypeAA

In [11]:
df = pd.read_csv(data_dir / 'joint_hd_b27pos.tsv', sep='\t')


FileNotFoundError: [Errno 2] No such file or directory: '/projects/immunestatus/emerson_hla/airr_format/joint_hd_b27pos.tsv'

In [ ]:
as_seqs = [
    'CASSVGLFSTDTQYF',
    'CASSVGLYSTDTQYF',
    'CASSAGLFSTDTQYF',
    'CASSAGLYSTDTQYF',
    'CASSLGLFSTDTQYF',
    'CASSLGLYSTDTQYF',
    'CASSPGLFSTDTQYF',
    'CASSPGLYSTDTQYF'
]
as_clonotypes = [ClonotypeAA(cdr3aa=x) for x in as_seqs]
as_clonotypes = [ClonotypeAA(cdr3aa=x) for x in as_seqs]
vdjdb = ClonotypeDataset(as_clonotypes)

In [ ]:
def has_vdjdb_match(x, threshold=1):
    return len(vdjdb.get_matching_clonotypes(x, threshold=threshold))

In [ ]:
df['as'] = df['junction_aa'].apply(has_vdjdb_match)

In [ ]:
df[df['as'] > 0]

In [ ]:
sum([len(x) for x in samples_desc])